# K=4 Cluster Count Justification
## Standalone notebook — loads `clean_fm_clusters.pkl` only

**Runs in ~25 seconds. No raw CSV files needed.**

### Why this matters
Choosing K is the clustering equivalent of choosing the number of layers in a neural network.
This notebook evaluates K=2–10 using multiple complementary diagnostic
lines of evidence. These diagnostics are not statistically independent
and are not treated as equivalent voting rules:

1. Internal clustering metrics across K
2. Inertia and elbow behaviour
3. GMM information criteria
4. Bootstrap resampling stability
5. Neighbouring K=3, K=4 and K=5 comparisons
6. Behavioural feature separability and interpretability

| Method | Neural Network Analogy |
|--------|------------------------|
| Multi-metric scores across K | Validation loss / accuracy curves |
| Bootstrap stability curve | Reproducibility across random seeds |
| Neighbour K comparison (K=3 vs K=4 vs K=5) | Ablation — what do you lose/gain? |
| Behavioural cross-tabulation | Qualitative model interpretability check |
| Consensus vote table | Ensemble of models agreeing on one answer |

**Place `clean_fm_clusters.pkl` in the same folder as this notebook before running.**

In [ ]:
# ── CELL 1: IMPORTS & DATA LOAD ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score
)
from sklearn.utils import resample

# ── Load pickle — the ONLY file this notebook needs ──────────────────────────
# Use the clean_fm_clusters.pkl regenerated by the corrected main pipeline.
clean_fm = pd.read_pickle('clean_fm_clusters.pkl')

CORE_FEATURES = [
    'evening_peak_ratio', 'night_baseload_ratio', 'load_uniformity',
    'load_factor', 'leisure_usage_ratio', 'log_daily_kwh'
]

OPTIMAL_K   = 4
RANDOM_SEED = 42
N_BOOT      = 100
k_range     = list(range(2, 11))

# ── Dynamic archetype name mapping ───────────────────────────────────────────
# KMeans numeric IDs are arbitrary and may swap after reruns.
# Therefore, names are derived from the median feature profile, not hard-coded.
cluster_profile = clean_fm.groupby('final_cluster')[CORE_FEATURES].median()

remaining = set(cluster_profile.index)

heavy_id = cluster_profile.loc[list(remaining), 'load_factor'].idxmax()
remaining.remove(heavy_id)

peak_id = cluster_profile.loc[list(remaining), 'evening_peak_ratio'].idxmax()
remaining.remove(peak_id)

daytime_id = cluster_profile.loc[list(remaining), 'night_baseload_ratio'].idxmin()
remaining.remove(daytime_id)

if len(remaining) != 1:
    raise RuntimeError(
        f"Dynamic archetype mapping failed. Remaining clusters: {remaining}"
    )

typical_id = list(remaining)[0]

c_names = {
    int(heavy_id): 'Heavy Continuous',
    int(peak_id): 'Peak-Exposed',
    int(daytime_id): 'Daytime-Active',
    int(typical_id): 'Typical Household'
}

ordered_archetypes = [
    'Peak-Exposed',
    'Typical Household',
    'Heavy Continuous',
    'Daytime-Active'
]

clean_fm['cluster_name'] = clean_fm['final_cluster'].map(c_names)

# Refit scaler on the locked feature matrix loaded from the regenerated pickle.
scaler  = StandardScaler()
X_clean = scaler.fit_transform(clean_fm[CORE_FEATURES])
orig_labels = clean_fm['final_cluster'].values
# Verify that the standalone notebook reconstructs the same
# scaled K=4 solution as the corrected thesis master.
alignment_labels = KMeans(
    n_clusters=4,
    random_state=RANDOM_SEED,
    n_init=20
).fit_predict(X_clean)

master_alignment_ari = adjusted_rand_score(
    orig_labels,
    alignment_labels
)

print(
    f"Alignment ARI with stored master labels: "
    f"{master_alignment_ari:.6f}"
)

assert np.isclose(
    master_alignment_ari,
    1.0,
    atol=1e-12
), (
    "The reconstructed K=4 solution does not match the "
    "stored corrected master labels."
)

print(f"Loaded: {len(clean_fm)} households | {len(CORE_FEATURES)} features")
print("Dynamic cluster-ID mapping:")
for cid in sorted(c_names):
    print(f"  cluster {cid}: {c_names[cid]}")

print("\nCluster distribution by archetype:")
print(clean_fm['cluster_name'].value_counts().reindex(ordered_archetypes).to_string())

print("\nReady.")


In [ ]:
# ── CELL 2: MULTI-METRIC SCORES ACROSS K=2–10 ────────────────────────────────
# Neural network analogy: validation loss curve.
# Watch silhouette, DB, CH, and GMM BIC across all K values.
# No single numerical criterion determines the final K.
# Numerical diagnostics are interpreted jointly with behavioural
# separability, stability and parsimony.
#
# IMPORTANT NOTE ON K=2:
# In this dataset, K=2 receives the strongest results for several compactness
# criteria. It is retained as a valid coarse diagnostic partition but is not
# selected for the final behavioural interpretation because two groups are
# insufficient to preserve the four substantively distinct usage profiles.
#
# The K>=3 comparison is therefore reported separately as an
# interpretation-constrained sensitivity analysis. It must not be presented
# as evidence that K=2 is mathematically invalid.
# ─────────────────────────────────────────────────────────────────────────────
print("Computing multi-metric scores K=2 to K=10...")

sil_scores, db_scores, ch_scores, inertias = [], [], [], []
bic_scores, aic_scores = [], []

for k in k_range:
    km     = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=20)
    labels = km.fit_predict(X_clean)
    sil_scores.append(silhouette_score(X_clean, labels))
    db_scores.append(davies_bouldin_score(X_clean, labels))
    ch_scores.append(calinski_harabasz_score(X_clean, labels))
    inertias.append(km.inertia_)

    gmm = GaussianMixture(
    n_components=k,
    covariance_type="full",
    random_state=RANDOM_SEED,
    n_init=10,
    max_iter=1000,
    reg_covar=1e-6 )
    gmm.fit(X_clean)
    if not gmm.converged_:
        raise RuntimeError(
        f"GMM did not converge for K={k} after "
        f"{gmm.n_iter_} iterations."
    )
    bic_scores.append(
    gmm.bic(X_clean)
    )
    aic_scores.append(
    gmm.aic(X_clean)
    )
    print(
    f"  K={k}: "
    f"Silhouette={sil_scores[-1]:.3f}  "
    f"DB={db_scores[-1]:.3f}  "
    f"CH={ch_scores[-1]:.0f}  "
    f"BIC={bic_scores[-1]:.0f}  "
    f"GMM converged={gmm.converged_}  "
    f"iterations={gmm.n_iter_}" 
    )

# ── Among K>=3: find local optimal K ─────────────────────────────────────────
sil_from3 = sil_scores[1:]   # index 0 = K=2, index 1 = K=3
db_from3  = db_scores[1:]
k_from3   = k_range[1:]

sil_local_opt = k_from3[np.argmax(sil_from3)]
db_local_opt  = k_from3[np.argmin(db_from3)]

print(
    "\nAmong K>=3 after retaining K=2 as a coarse "
    "diagnostic alternative:"
)
print(f"  Silhouette local max at K={sil_local_opt}  "
      f"(value={max(sil_from3):.3f})")
print(f"  Davies-Bouldin local min at K={db_local_opt}  "
      f"(value={min(db_from3):.3f})")

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Multi-Metric K Selection — K=2 to K=10\n"
             "(K=4 retained as an interpretation-led minimum sufficient solution)",
             fontsize=13, fontweight='bold')

# Silhouette
ax = axes[0, 0]
ax.plot(k_range, sil_scores, marker='o', color='#2c3e50', linewidth=2)
ax.axvline(OPTIMAL_K, color='red', linestyle='--', linewidth=1.5,
           label=f'K={OPTIMAL_K} (selected)')
ax.axvspan(2.5, 11, alpha=0.05, color='green',
           label='Interpretable range (K≥3)')
ax.set_title("Silhouette Score (higher = better)")
ax.set_xlabel("K"); ax.set_ylabel("Score")
ax.set_xticks(k_range); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Davies-Bouldin
ax = axes[0, 1]
ax.plot(k_range, db_scores, marker='o', color='#e74c3c', linewidth=2)
ax.axvline(OPTIMAL_K, color='red', linestyle='--', linewidth=1.5,
           label=f'K={OPTIMAL_K} (selected)')
ax.axvspan(2.5, 11, alpha=0.05, color='green')
ax.set_title("Davies-Bouldin Index (lower = better)")
ax.set_xlabel("K"); ax.set_ylabel("DB Index")
ax.set_xticks(k_range); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Inertia (elbow)
ax = axes[1, 0]
ax.plot(k_range, inertias, marker='o', color='#27ae60', linewidth=2)
ax.axvline(OPTIMAL_K, color='red', linestyle='--', linewidth=1.5,
           label=f'K={OPTIMAL_K} (selected)')
ax.set_title("Elbow Method — Inertia")
ax.set_xlabel("K"); ax.set_ylabel("Within-cluster SS")
ax.set_xticks(k_range); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# GMM BIC
ax = axes[1, 1]
ax.plot(k_range, bic_scores, marker='o', color='#d35400',
        linewidth=2, label='BIC')
ax.plot(k_range, aic_scores, marker='s', color='#f39c12',
        linewidth=2, linestyle='--', label='AIC')
ax.axvline(OPTIMAL_K, color='red', linestyle='--', linewidth=1.5,
           label=f'K={OPTIMAL_K} (selected)')
ax.set_title("GMM BIC / AIC — Model Selection")
ax.set_xlabel("K"); ax.set_ylabel("Criterion value")
ax.set_xticks(k_range); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 3: BOOTSTRAP STABILITY CURVE ACROSS ALL K ───────────────────────────
# For each K, run bootstrap iterations and measure mean ARI against a reference
# clustering fitted on the full locked six-feature matrix.
# This is a stability diagnostic only; it does not by itself determine final K.
# ─────────────────────────────────────────────────────────────────────────────

print(f"Computing bootstrap stability ({N_BOOT} iterations per K)...")

stab_means, stab_mins, stab_maxs = [], [], []

for k in k_range:
    km_ref = KMeans(
        n_clusters=k,
        random_state=RANDOM_SEED,
        n_init=20
    )
    ref_labels = km_ref.fit_predict(X_clean)

    ari_list = []

    for i in range(N_BOOT):
        X_boot = resample(
            X_clean,
            random_state=i
        )

        km_b = KMeans(
            n_clusters=k,
            random_state=i,
            n_init=20
        )

        km_b.fit(X_boot)
        boot_labels = km_b.predict(X_clean)

        ari_list.append(
            adjusted_rand_score(ref_labels, boot_labels)
        )

    stab_means.append(np.mean(ari_list))
    stab_mins.append(np.min(ari_list))
    stab_maxs.append(np.max(ari_list))

    print(
        f"  K={k}: mean ARI={stab_means[-1]:.3f}  "
        f"min={stab_mins[-1]:.3f}  max={stab_maxs[-1]:.3f}"
    )

stab_from3 = stab_means[1:]
stab_opt_from3 = k_from3[np.argmax(stab_from3)]

print(
    f"\nAmong K>=3: most stable K = {stab_opt_from3} "
    f"(mean ARI={max(stab_from3):.3f})"
)

print(
    f"Selected K={OPTIMAL_K} bootstrap stability = "
    f"{stab_means[OPTIMAL_K - 2]:.3f}"
)

if stab_opt_from3 != OPTIMAL_K:
    print(
        "Bootstrap stability does not select K=4. "
        "It is reported as a diagnostic, while K=4 is retained based on "
        "interpretability and preservation of the Peak-Exposed archetype."
    )

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    k_range,
    stab_means,
    marker="o",
    color="#27ae60",
    linewidth=2.5,
    label="Mean ARI"
)

ax.fill_between(
    k_range,
    stab_mins,
    stab_maxs,
    alpha=0.15,
    color="#27ae60",
    label="Min–Max range"
)

ax.axvline(
    OPTIMAL_K,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label=f"K={OPTIMAL_K} retained"
)

ax.axvspan(
    1.5,
    2.5,
    alpha=0.12,
    color="grey",
    label="K=2: too coarse for final interpretation"
)

ax.set_title(
    f"Bootstrap Stability Curve ({N_BOOT} iterations per K)\n"
    "Higher ARI indicates stronger resampling stability",
    fontsize=12
)

ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("Adjusted Rand Index (ARI)")
ax.set_xticks(k_range)
ax.legend(fontsize=9)
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 4: NEIGHBOUR K COMPARISON — K=3, K=4, K=5 ───────────────────────────
# Checks what happens immediately below and above the locked K=4 solution.
# This is an interpretation diagnostic; it does not change the final labels.
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 68)
print("  NEIGHBOUR K COMPARISON: K=3, K=4, K=5")
print("=" * 68)

km3 = KMeans(
    n_clusters=3,
    random_state=RANDOM_SEED,
    n_init=20
)

km5 = KMeans(
    n_clusters=5,
    random_state=RANDOM_SEED,
    n_init=20
)

labels3 = km3.fit_predict(X_clean)
labels5 = km5.fit_predict(X_clean)

df = clean_fm.copy()
df["k3"] = labels3
df["k5"] = labels5

print("\n── K=3 vs K=4 Cross-Tabulation ─────────────────────────────────")
print("   Where do K=4 archetypes land when K=3 is used?")

ct3 = pd.crosstab(df["k3"], df["cluster_name"])
ct3 = ct3.reindex(columns=ordered_archetypes, fill_value=0)
ct3.index = [f"K3 cluster {i}" for i in ct3.index]

print(ct3.to_string())

print("\n   K=3 centroid profiles:")
print(
    df.groupby("k3")[CORE_FEATURES]
    .median()
    .round(3)
    .to_string()
)

print("\n   Interpretation:")
pe_counts = ct3["Peak-Exposed"]
main_absorber = pe_counts.idxmax()
n_pe_total = int(ct3["Peak-Exposed"].sum())

print(
    f"   The {n_pe_total} Peak-Exposed households from the locked K=4 "
    "solution are distributed across broader K=3 clusters."
)

print(
    f"   The largest share is absorbed into {main_absorber}, reducing the "
    "visibility of Peak-Exposed as a separate archetype."
)

print(
    "   Therefore, K=3 is considered too coarse for the final behavioural "
    "segmentation because it does not preserve the Peak-Exposed pattern "
    "as a separate group."
)

print("\n── K=5 vs K=4 Cross-Tabulation ─────────────────────────────────")
print("   What does K=5 add over K=4?")

ct5 = pd.crosstab(df["k5"], df["cluster_name"])
ct5 = ct5.reindex(columns=ordered_archetypes, fill_value=0)
ct5.index = [f"K5 cluster {i}" for i in ct5.index]

print(ct5.to_string())

print("\n   K=5 centroid profiles:")
print(
    df.groupby("k5")[CORE_FEATURES]
    .median()
    .round(3)
    .to_string()
)

print("\n   Interpretation:")
print(
    "   K=5 introduces additional subdivisions and some mixing of the "
    "locked K=4 archetypes, especially around Typical Household and "
    "Heavy Continuous households."
)

print(
    "   The extra cluster does not provide a clearly necessary new "
    "behavioural archetype for the final interpretation."
)

print(
    "   Therefore, K=5 is treated as a more granular alternative, but "
    "not retained as the final solution."
)

print("\n── Summary ───────────────────────────────────────────────")
print(
    "  K=3: Too coarse; Peak-Exposed is not preserved as a separate archetype."
)
print(
    "  K=4: Retained as the minimum sufficient interpretable solution."
)
print(
    "  K=5: Adds subdivisions without a clearly necessary new behavioural archetype."
)


In [ ]:
# ── CELL 5: FEATURE SEPARABILITY BY K ────────────────────────────────────────
# Check whether the Evening Peak Ratio — the defining feature of the
# Peak-Exposed archetype — is separable at K=3 vs K=4.
# If EPR is not separable at K=3, the archetype is statistically less visible.
# ─────────────────────────────────────────────────────────────────────────────
from scipy.stats import kruskal

print("=" * 68)
print("  EPR SEPARABILITY: K=3 vs K=4")
print("  (Is the Peak-Exposed signature statistically visible at each K?)")
print("=" * 68)

df['k3'] = labels3

for label, col in [('K=3', 'k3'), ('K=4', 'final_cluster')]:
    groups = [
        df[df[col] == c]['evening_peak_ratio'].values
        for c in df[col].unique()
    ]
    stat, p = kruskal(*groups)
    print(f"\n  {label} — Kruskal-Wallis on Evening Peak Ratio:")
    print(f"    H={stat:.2f}  p={p:.2e}")
    print(f"    Cluster EPR medians:")
    epr = df.groupby(col)['evening_peak_ratio'].median().round(3)
    print(
        f"    Range: {epr.min():.3f} to {epr.max():.3f}  "
        f"(spread = {epr.max() - epr.min():.3f})"
    )
    if col == "final_cluster":
        epr_named = (
            df.groupby("cluster_name")['evening_peak_ratio']
            .median()
            .reindex(ordered_archetypes)
            .round(3)
        )
        print(f"    {epr_named.to_dict()}")
    else:
        print(f"    {epr.to_dict()}")

epr_named = (
    df.groupby("cluster_name")['evening_peak_ratio']
    .median()
    .reindex(ordered_archetypes)
    .round(3)
)

peak_epr = epr_named["Peak-Exposed"]
typical_epr = epr_named["Typical Household"]

print("\n  INTERPRETATION:")
print(
    "  At K=4, the EPR spread across clusters is large — Peak-Exposed "
    f"(EPR={peak_epr:.3f}) is separated from Typical Household "
    f"(EPR={typical_epr:.3f})."
)
print(
    "  At K=3, this spread is reduced because Peak-Exposed is absorbed "
    "into a broader cluster, diluting the EPR signal that defines the archetype."
)

# Boxplot comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    "Evening Peak Ratio Distribution by Cluster\n"
    "K=3 vs K=4 — Does the Peak-Exposed signature survive?",
    fontsize=12,
    fontweight='bold'
)

colors3 = ['#3498db', '#e74c3c', '#2ecc71']
colors4 = ['#e74c3c', '#3498db', '#2ecc71', '#95a5a6']

# K=3 boxplot
ax = axes[0]
data3 = [
    df[df['k3'] == c]['evening_peak_ratio'].values
    for c in sorted(df['k3'].unique())
]
bp3 = ax.boxplot(data3, patch_artist=True, notch=False)
for patch, color in zip(bp3['boxes'], colors3):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title(
    "K=3: Peak-Exposed signal is reduced\n"
    "(absorbed into larger clusters)"
)
ax.set_xlabel("K=3 Cluster")
ax.set_ylabel("Evening Peak Ratio")
ax.set_xticklabels([f'K3-{i}' for i in sorted(df['k3'].unique())])
ax.grid(True, axis='y', alpha=0.3)

# K=4 boxplot
ax = axes[1]
sorted_final_clusters = sorted(df['final_cluster'].unique())
data4 = [
    df[df['final_cluster'] == c]['evening_peak_ratio'].values
    for c in sorted_final_clusters
]
bp4 = ax.boxplot(data4, patch_artist=True, notch=False)
for patch, color in zip(bp4['boxes'], colors4):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title(
    "K=4: Peak-Exposed clearly separated\n"
    f"(EPR={peak_epr:.3f} vs {typical_epr:.3f} for Typical)"
)
ax.set_xlabel("K=4 Cluster")
ax.set_ylabel("Evening Peak Ratio")
ax.set_xticklabels(
    [c_names[c] for c in sorted_final_clusters],
    rotation=12,
    ha='right'
)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── CELL 6: CONSENSUS VOTE TABLE ─────────────────────────────────────────────
# Neural network analogy: ensemble of models voting on one answer.
# Six diagnostic criteria are compared. They are not treated as
# statistically independent votes or as equivalent decision rules.
# We evaluate votes both globally (all K) and locally (K>=3 only),
#K=2 is not retained for the final behavioural interpretation
#because it is too coarse to represent the four required usage profiles.
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 68)
print("  CONSENSUS VOTE TABLE")
print("  Six diagnostic criteria — preferred K under each criterion")
print("=" * 68)

# Compute second derivative of inertia for elbow
inertia_arr  = np.array(inertias)
second_deriv = np.diff(np.diff(inertia_arr))
# +1 because second diff loses 2 points, +1 for 0-indexing to k_range
elbow_k_global = k_range[np.argmax(second_deriv) + 1]

# Restrict second deriv to K>=3 (indices 1 onward in k_range)
second_deriv_from3 = second_deriv[0:]   # second_deriv already starts at K=3
elbow_k_local = k_range[np.argmax(second_deriv_from3) + 1]

# Global votes (all K)
sil_vote_g    = k_range[np.argmax(sil_scores)]
db_vote_g     = k_range[np.argmin(db_scores)]
ch_vote_g     = k_range[np.argmax(ch_scores)]
stab_vote_g   = k_range[np.argmax(stab_means)]
bic_vote_g    = k_range[np.argmin(bic_scores)]

# Local votes (K>=3)
sil_vote_l    = k_from3[np.argmax(sil_from3)]
db_vote_l     = k_from3[np.argmin(db_from3)]
ch_vote_l     = k_from3[np.argmax(ch_scores[1:])]
stab_vote_l   = k_from3[np.argmax(stab_means[1:])]
bic_vote_l    = k_from3[np.argmin(bic_scores[1:])]

criteria = [
    ("Silhouette Score (max)",        sil_vote_g,   sil_vote_l),
    ("Davies-Bouldin Index (min)",    db_vote_g,    db_vote_l),
    ("Calinski-Harabasz (max)",       ch_vote_g,    ch_vote_l),
    ("Elbow / 2nd derivative",        elbow_k_global, elbow_k_local),
    ("Bootstrap Stability (max ARI)", stab_vote_g,  stab_vote_l),
    ("GMM BIC (min)",                 bic_vote_g,   bic_vote_l),
]

print(f"\n  {'Criterion':<34} {'Global vote':>12} {'K>=3 vote':>12}  "
      f"{'Selects K=4?':>14}")
print("  " + "-" * 78)

global_agree = 0
local_agree  = 0
for name, g_vote, l_vote in criteria:
    g_flag = "✓" if g_vote == OPTIMAL_K else f"K={g_vote}"
    l_flag = "✓" if l_vote == OPTIMAL_K else f"K={l_vote}"
    correct = "✓ YES" if l_vote == OPTIMAL_K else "✗ NO"
    if g_vote == OPTIMAL_K: global_agree += 1
    if l_vote == OPTIMAL_K: local_agree  += 1
    print(f"  {name:<34} {str(g_vote):>12} {str(l_vote):>12}  {correct:>14}")

print(f"\n  Global consensus (all K):      {global_agree}/{len(criteria)} methods select K={OPTIMAL_K}")
print(f"  Local consensus (K>=3 only):   {local_agree}/{len(criteria)} methods select K={OPTIMAL_K}")

print("  CONSERVATIVE INTERPRETATION")
print("=" * 68)

print(f"""
  The diagnostic criteria do not unanimously select K={OPTIMAL_K}.
  Several unrestricted criteria favour K=2, while some criteria favour
  higher K values. Therefore, K=4 is not claimed as a purely
  metric-optimal solution.

  K=2 is excluded because it is too coarse for the intended behavioural
  segmentation. K=3 reduces the visibility of the Peak-Exposed pattern,
  while K=5 introduces additional subdivisions without a clearly necessary
  new behavioural archetype.

  K=4 is retained as the minimum sufficient interpretable solution:
  it preserves the four main behavioural archetypes while avoiding
  unnecessary extra granularity.
""")

In [ ]:
# ── CELL 7: SUMMARY FIGURE — ALL EVIDENCE IN ONE PLOT ────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

fig.suptitle(
    "K=4 Justification Dashboard — Six Diagnostic Lines of Evidence\n"
"(Red dashed line = retained K=4; grey band = coarse K=2 alternative)",
    fontsize=13, fontweight='bold'
)

def add_vline_band(ax):
    ax.axvline(OPTIMAL_K, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.axvspan(1.5, 2.5, alpha=0.10, color='grey')
    ax.set_xticks(k_range)
    ax.grid(True, linestyle='--', alpha=0.3)

# 1. Silhouette
ax = fig.add_subplot(gs[0, 0])
ax.plot(k_range, sil_scores, marker='o', color='#2c3e50', linewidth=2)
add_vline_band(ax)
ax.set_title("① Silhouette (max = better)\nLocal optimum at K=4 among K≥3")
ax.set_xlabel("K"); ax.set_ylabel("Score")

# 2. Davies-Bouldin
ax = fig.add_subplot(gs[0, 1])
ax.plot(k_range, db_scores, marker='o', color='#e74c3c', linewidth=2)
add_vline_band(ax)
ax.set_title("Davies-Bouldin score across K")
ax.set_xlabel("K"); ax.set_ylabel("DB Index")

# 3. Bootstrap stability
ax = fig.add_subplot(gs[0, 2])
ax.plot(k_range, stab_means, marker='o', color='#27ae60', linewidth=2)
ax.fill_between(k_range, stab_mins, stab_maxs, alpha=0.15, color='#27ae60')
add_vline_band(ax)
ax.set_title("Bootstrap stability across K")
ax.set_xlabel("K"); ax.set_ylabel("Mean ARI")

# 4. EPR separability comparison (K=3 vs K=4)
ax = fig.add_subplot(gs[1, 0])
epr_k3 = [df[df['k3'] == c]['evening_peak_ratio'].values
           for c in sorted(df['k3'].unique())]
epr_k4 = [df[df['final_cluster'] == c]['evening_peak_ratio'].values
           for c in sorted(df['final_cluster'].unique())]
positions3 = [1, 2, 3]
positions4 = [5, 6, 7, 8]
bp3 = ax.boxplot(epr_k3, positions=positions3, widths=0.6, patch_artist=True,
                 boxprops=dict(facecolor='#3498db', alpha=0.6))
bp4 = ax.boxplot(epr_k4, positions=positions4, widths=0.6, patch_artist=True,
                 boxprops=dict(facecolor='#e74c3c', alpha=0.6))
ax.set_xticks([2, 6.5])
ax.set_xticklabels([
    'K=3\n(PE not separate)',
    'K=4\n(PE separated)'
])
ax.set_title(
    "④ EPR Separability\n"
    "K=4 preserves Peak-Exposed as a separate group"
)
ax.set_ylabel("Evening Peak Ratio")
ax.grid(True, axis='y', alpha=0.3)

# 5. Inertia elbow
ax = fig.add_subplot(gs[1, 1])
ax.plot(k_range, inertias, marker='o', color='#8e44ad', linewidth=2)
add_vline_band(ax)
ax.set_title("Elbow / WCSS across K")
ax.set_xlabel("K"); ax.set_ylabel("Inertia (WCSS)")

# 6. GMM BIC
ax = fig.add_subplot(gs[1, 2])
ax.plot(k_range, bic_scores, marker='o', color='#d35400', linewidth=2, label='BIC')
ax.plot(k_range, aic_scores, marker='s', color='#f39c12',
        linewidth=2, linestyle='--', label='AIC')
add_vline_band(ax)
ax.set_title("GMM BIC/AIC across K")
ax.set_xlabel("K"); ax.set_ylabel("Criterion")
ax.legend(fontsize=8)

plt.savefig('K4_justification_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("Dashboard saved as K4_justification_dashboard.png")
print(
    "Safe interpretation: K=4 is retained as an interpretation-led minimum "
    "sufficient solution. It preserves the four useful household archetypes, "
    "but it is not selected by every numerical diagnostic."
)

In [ ]:
# ============================================================
# FINAL CONSERVATIVE K=4 DECISION AUDIT
# Does not refit or change the locked K=4 labels
# ============================================================

import numpy as np
import pandas as pd

required_objects = [
    "k_range",
    "sil_scores",
    "db_scores",
    "ch_scores",
    "bic_scores",
    "stab_means",
    "inertias"
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Run Cells 1-6 first. Missing objects: {missing}"
    )

k_values = np.asarray(k_range)
inertia_values = np.asarray(inertias)

second_difference = np.diff(
    np.diff(inertia_values)
)

elbow_k = int(
    k_values[
        np.argmax(second_difference) + 1
    ]
)

decision_table = pd.DataFrame({
    "criterion": [
        "Silhouette (maximum)",
        "Davies-Bouldin (minimum)",
        "Calinski-Harabasz (maximum)",
        "Elbow / second difference",
        "Bootstrap mean ARI (maximum)",
        "GMM BIC (minimum)"
    ],
    "preferred_k_all_tested": [
        int(k_values[np.argmax(sil_scores)]),
        int(k_values[np.argmin(db_scores)]),
        int(k_values[np.argmax(ch_scores)]),
        elbow_k,
        int(k_values[np.argmax(stab_means)]),
        int(k_values[np.argmin(bic_scores)])
    ]
})

decision_table["selects_k4"] = (
    decision_table["preferred_k_all_tested"] == 4
)

print("Actual criterion winners:")
print(decision_table.to_string(index=False))

print(
    f"\nCriteria selecting K=4: "
    f"{decision_table['selects_k4'].sum()} "
    f"of {len(decision_table)}"
)

decision_table.to_csv(
    "K4_final_conservative_decision_audit.csv",
    index=False
)

print("\nFinal defensible decision:")

print(
    "K=4 is retained as the minimum sufficient "
    "interpretable solution, not because every diagnostic "
    "selects K=4. K=3 is too coarse to preserve the "
    "Peak-Exposed archetype, while K=5 mainly subdivides "
    "existing archetypes. K=4 also provides the strongest "
    "silhouette result among K>=3."
)

In [ ]:
# ============================================================
# CORRECTED K-SELECTION SUMMARY
# Reports unrestricted and K>=3 results separately
# ============================================================

import numpy as np
import pandas as pd

k_values = np.asarray(
    k_range,
    dtype=int
)

sil_values = np.asarray(
    sil_scores,
    dtype=float
)

db_values = np.asarray(
    db_scores,
    dtype=float
)

ch_values = np.asarray(
    ch_scores,
    dtype=float
)

bic_values = np.asarray(
    bic_scores,
    dtype=float
)

stability_values = np.asarray(
    stab_means,
    dtype=float
)

inertia_values = np.asarray(
    inertias,
    dtype=float
)

eligible_mask = k_values >= 3

eligible_k = k_values[eligible_mask]

def preferred_k(
    values,
    mode,
    mask=None
):
    values = np.asarray(
        values,
        dtype=float
    )

    if mask is None:
        selected_values = values
        selected_k = k_values
    else:
        selected_values = values[mask]
        selected_k = k_values[mask]

    if mode == "max":
        return int(
            selected_k[
                np.nanargmax(selected_values)
            ]
        )

    if mode == "min":
        return int(
            selected_k[
                np.nanargmin(selected_values)
            ]
        )

    raise ValueError(
        "mode must be 'max' or 'min'"
    )

# Elbow candidates correspond to the central K values
second_difference = np.diff(
    inertia_values,
    n=2
)

central_k = k_values[1:-1]

all_elbow_k = int(
    central_k[
        np.argmax(second_difference)
    ]
)

eligible_elbow_mask = central_k >= 3

eligible_elbow_k = int(
    central_k[eligible_elbow_mask][
        np.argmax(
            second_difference[
                eligible_elbow_mask
            ]
        )
    ]
)

summary = pd.DataFrame({
    "criterion": [
        "Silhouette maximum",
        "Davies-Bouldin minimum",
        "Calinski-Harabasz maximum",
        "Elbow / second difference",
        "Bootstrap mean ARI maximum",
        "GMM BIC minimum"
    ],
    "winner_all_tested": [
        preferred_k(
            sil_values,
            "max"
        ),
        preferred_k(
            db_values,
            "min"
        ),
        preferred_k(
            ch_values,
            "max"
        ),
        all_elbow_k,
        preferred_k(
            stability_values,
            "max"
        ),
        preferred_k(
            bic_values,
            "min"
        )
    ],
    "winner_among_k_ge_3": [
        preferred_k(
            sil_values,
            "max",
            eligible_mask
        ),
        preferred_k(
            db_values,
            "min",
            eligible_mask
        ),
        preferred_k(
            ch_values,
            "max",
            eligible_mask
        ),
        eligible_elbow_k,
        preferred_k(
            stability_values,
            "max",
            eligible_mask
        ),
        preferred_k(
            bic_values,
            "min",
            eligible_mask
        )
    ]
})

summary["k4_winner_all"] = (
    summary["winner_all_tested"] == 4
)

summary["k4_winner_among_k_ge_3"] = (
    summary["winner_among_k_ge_3"] == 4
)

print("Corrected criterion summary:")
print(
    summary.to_string(index=False)
)

k4_index = np.where(
    k_values == 4
)[0][0]

print("\nK=4 diagnostic values:")
print(
    f"Silhouette: "
    f"{sil_values[k4_index]:.4f}"
)
print(
    f"Davies-Bouldin: "
    f"{db_values[k4_index]:.4f}"
)
print(
    f"Calinski-Harabasz: "
    f"{ch_values[k4_index]:.4f}"
)
print(
    f"Bootstrap mean ARI: "
    f"{stability_values[k4_index]:.4f}"
)
print(
    f"GMM BIC: "
    f"{bic_values[k4_index]:.4f}"
)

print(
    "\nCriteria selecting K=4 among K>=3:",
    int(
        summary[
            "k4_winner_among_k_ge_3"
        ].sum()
    ),
    "of",
    len(summary)
)

summary.to_csv(
    "K_selection_corrected_unrestricted_and_K_ge_3.csv",
    index=False
)

print("\nFinal interpretation:")
print(
    "K=2 provides the strongest result for several "
    "unrestricted numerical criteria was retained as a coarse diagnostic alternative but was not"
    "selected for the final behavioural interpretation. Among the "
    "remaining candidate solutions, K=4 is retained as an "
    "interpretation-led minimum sufficient solution. The "
    "diagnostic criteria do not unanimously select K=4."
)
